# 🚀 AIE4ML Tutorial 2: Branching + Permute + Multi-output

This short tutorial focuses on one model that stresses key architectural features:
- internal fanout / broadcasting after a shared dense layer
- `Permute((2, 1))` in one branch
- two output tensors
- ND dense path (Dense on rank-3 tensors)

## Setup & Imports

```bash
conda create -n aie4ml_env python=3.10 -y && conda activate aie4ml_env
pip install "tensorflow==2.15.*" "tensorflow-model-optimization==0.7.5" qkeras hls4ml
pip install aie4ml pyparsing ipykernel pydot graphviz
```
```

In [ ]:
import numpy as np
import subprocess

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Permute
from tensorflow.keras.optimizers import Adam

from qkeras import QDense, QActivation, quantized_bits, quantized_relu

import hls4ml

## 1️⃣ Build the QKeras model


In [ ]:
DIM2 = 16
DIM3 = 128

BASE = 64
B1_MID0 = 32
B1_MID1 = 64
B1_OUT = 128

B2_MID0 = 64
B2_MID1 = 64
B2_OUT = 256


def build_qkeras_model():
    inputs = Input(shape=(DIM2, DIM3), name="input_layer")

    x = QActivation(quantized_bits(16, 6), name="input_quant")(inputs)

    # Shared trunk
    x = QDense(
        BASE,
        name="base_qfc0",
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 2, alpha=1),
    )(x)
    x = QActivation(quantized_relu(8), name="base_qrelu0")(x)

    # Branch 1
    b1 = QDense(
        B1_MID0,
        name="branch1_qfc1",
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 2, alpha=1),
    )(x)
    b1 = QActivation(quantized_relu(8), name="branch1_qrelu1")(b1)

    b1 = Permute((2, 1), name="branch1_permute")(b1)

    b1 = QDense(
        B1_MID1,
        name="branch1_qfc2",
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 2, alpha=1),
    )(b1)
    b1 = QActivation(quantized_relu(8), name="branch1_qrelu2")(b1)

    b1 = QDense(
        B1_OUT,
        name="branch1_qfc3",
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 2, alpha=1),
    )(b1)
    b1 = QActivation(quantized_bits(8, 2), name="branch1_output_quant")(b1)

    # Branch 2
    b2 = QDense(
        B2_MID0,
        name="branch2_qfc1",
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 2, alpha=1),
    )(x)
    b2 = QActivation(quantized_relu(8), name="branch2_qrelu1")(b2)

    b2 = QDense(
        B2_MID1,
        name="branch2_qfc2",
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 2, alpha=1),
    )(b2)
    b2 = QActivation(quantized_relu(8), name="branch2_qrelu2")(b2)

    b2 = QDense(
        B2_OUT,
        name="branch2_qfc3",
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 2, alpha=1),
    )(b2)
    b2 = QActivation(quantized_bits(8, 2), name="branch2_output_quant")(b2)

    model = Model(inputs=inputs, outputs=[b1, b2], name="balanced_two_branch_qkeras_model")
    return model



model = build_qkeras_model()
model.compile(optimizer=Adam(1e-3), loss="mse")
model.summary()


## 2️⃣ Convert to AIE backend

In [ ]:
cfg = hls4ml.utils.config_from_keras_model(model, granularity="name")

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=cfg,
    output_dir="proj_hls_t2",
    project_name="proj_hls_t2",
    bit_exact=True,
)

aie_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=cfg,
    output_dir="proj_aie_t2",
    project_name="proj_aie_t2",
    backend="aie",
    batch_size=1,
    iterations=10,
)

## 3️⃣ Visualize the model

#### Architecture decisions highlighted by this model

- **Input prec.**: 16b precision on input
- **Fanout/broadcast after `qfc0`**: shared tensor consumed by two branches.
- **Permute in one branch**: requires explicit layout/view handling in staging/memory descriptors.
- **Two graph outputs**: requires stable output-port mapping and simulator output decoding per tensor.
- **ND Dense path**: Dense on rank-3 tensors stresses descriptor shape/traversal contracts.

These are not only layer conversions; they are system-level choices in lowering, memory planning, and graph I/O contracts.


In [ ]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

## 4️⃣  Compile and run x86 inference


In [ ]:
aie_model.compile()

x = np.random.random((1, DIM2, DIM3)).astype(np.float32)

y_qkeras = model.predict(x)
y_aie = aie_model.predict(x, simulator="x86")

max_diff = [
    np.max(np.abs(a - b))
    for a, b in zip(y_aie.values(), y_qkeras)
]

print(max_diff)

## 5️⃣ Compile AIE


In [ ]:
print("Building AIE project...")

aie_model.build()

print("AIE build & compile completed.")

## 4️⃣  Run AIE inference and test bit-exactness


In [ ]:
x = np.random.random((1, DIM2, DIM3)).astype(np.float32)

y_qkeras = model.predict(x)
y_aie = aie_model.predict(x, simulator="aie")

max_diff = [
    np.max(np.abs(a - b))
    for a, b in zip(y_aie.values(), y_qkeras)
]

print(max_diff)

# 7️⃣ View AIE Simulation Report


In [ ]:
from aie4ml.simulation import read_aie_report

report = read_aie_report(aie_model)
report

# 🎉 Tutorial Complete!

